In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/content/Gold-Silver-GeopoliticalRisk_HistoricalData.csv')
df = df.iloc[:, :-1] # the last column only consisted of NaNs so I got rid of it
df.head()


In [ ]:
df.dropna(inplace=True) # drops NaNs from the csv

In [ ]:
# makes date column into datetime so it is easier for pandas to read
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(by='DATE').reset_index(drop=True)

#lag variable by 1 day
df['GOLD_PRICE_LAG1'] = df['GOLD_PRICE'].shift(1)
df['SILVER_PRICE_LAG1'] = df['SILVER_PRICE'].shift(1)

# lag by 2 days
df['GOLD_PRICE_LAG2'] = df['GOLD_PRICE'].shift(2)
df['SILVER_PRICE_LAG2'] = df['SILVER_PRICE'].shift(2)


df['MONTH'] = df['DATE'].dt.month
df['DAY_OF_WEEK'] = df['DATE'].dt.dayofweek
df['DAY_OF_YEAR'] = df['DATE'].dt.dayofyear

# Drop rows with NaN values created by shifting (the first few rows)
df.dropna(inplace=True)

# target variables that the RF wil be trained to predict
y = df[['GOLD_PRICE', 'SILVER_PRICE']].copy()
y['GSR'] = y['GOLD_PRICE'] / y['SILVER_PRICE']


X = df[['GPRD', 'GPRD_ACT', 'GPRD_THREAT',
        'GOLD_PRICE_LAG1', 'SILVER_PRICE_LAG1',
        'GOLD_PRICE_LAG2', 'SILVER_PRICE_LAG2',
        'MONTH', 'DAY_OF_WEEK', 'DAY_OF_YEAR']].copy()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, random_state = 67, test_size = .2)

rfr = RandomForestRegressor(n_estimators=1000, max_depth=12,
                            min_samples_split= 25, min_samples_leaf=2,
                            max_features='sqrt', random_state=67,
                                                        n_jobs=-1)

In [ ]:
rfr.fit(X_train, y_train)

In [ ]:
y_pred = rfr.predict(X_test)

In [ ]:
rfr.score(X_test, y_test) * 100

In [ ]:

mae_gold = mean_absolute_error(y_test['GOLD_PRICE'], y_pred[:, 0])
mae_silver = mean_absolute_error(y_test['SILVER_PRICE'], y_pred[:, 1])
mae_gsr = mean_absolute_error(y_test['GSR'], y_pred[:, 2])

print(f"Mean Absolute Error for GOLD_PRICE: {mae_gold:.2f}")
print(f"Mean Absolute Error for SILVER_PRICE: {mae_silver:.2f}")
print(f"Mean Absolute Error for GSR: {mae_gsr:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Actual vs. Predicted Values', fontsize=16)

sns.scatterplot(x=y_test['GOLD_PRICE'], y=y_pred[:, 0], ax=axes[0], alpha=0.6)
axes[0].set_title('GOLD_PRICE')
axes[0].set_xlabel('Actual Gold Price')
axes[0].set_ylabel('Predicted Gold Price')
axes[0].plot([y_test['GOLD_PRICE'].min(), y_test['GOLD_PRICE'].max()],
             [y_test['GOLD_PRICE'].min(), y_test['GOLD_PRICE'].max()], 'r--') # Reference line

sns.scatterplot(x=y_test['SILVER_PRICE'], y=y_pred[:, 1], ax=axes[1], alpha=0.6)
axes[1].set_title('SILVER_PRICE')
axes[1].set_xlabel('Actual Silver Price')
axes[1].set_ylabel('Predicted Silver Price')
axes[1].plot([y_test['SILVER_PRICE'].min(), y_test['SILVER_PRICE'].max()],
             [y_test['SILVER_PRICE'].min(), y_test['SILVER_PRICE'].max()], 'r--') # Reference line

sns.scatterplot(x=y_test['GSR'], y=y_pred[:, 2], ax=axes[2], alpha=0.6)
axes[2].set_title('GSR')
axes[2].set_xlabel('Actual GSR')
axes[2].set_ylabel('Predicted GSR')
axes[2].plot([y_test['GSR'].min(), y_test['GSR'].max()],
             [y_test['GSR'].min(), y_test['GSR'].max()], 'r--') # Reference line

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()